# 真实集成：响应级余弦一致性

真实 API 采样完整回答，本地 SentenceTransformer 对整条响应与每条采样计算余弦一致性。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai,embeddings]"`。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `LOCAL_EMBEDDING_MODEL: str`：本地 SentenceTransformer 目录。
- `EMBEDDING_DEVICE: str = 'cpu'`：embedding 设备。
- `NUM_SAMPLES: int = 5`：采样数。
- `TEMPERATURE: float = 0.8`：API 采样多样性。

## 步骤 1：配置真实生成和 embedding

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

from hallucination.detection.backends.embeddings import SentenceTransformerEmbeddingBackend
LOCAL_EMBEDDING_MODEL = "/absolute/path/to/your/sentence-transformer"
EMBEDDING_DEVICE = "cpu"
if not Path(LOCAL_EMBEDDING_MODEL).is_dir():
    raise ValueError("Set LOCAL_EMBEDDING_MODEL to an existing local embedding model directory.")
embedding = SentenceTransformerEmbeddingBackend(LOCAL_EMBEDDING_MODEL, device=EMBEDDING_DEVICE)

## 步骤 2：运行 scorer

In [ ]:
from hallucination.detection.blackbox.response_cosine import ResponseCosineConsistencyScorer
from hallucination.detection.pipelines.short_answer import BlackBoxSamplingPipeline
NUM_SAMPLES = 5
scorer = ResponseCosineConsistencyScorer(embedding, normalize=True)
result = BlackBoxSamplingPipeline(backend, scorers=[scorer], num_samples=NUM_SAMPLES).detect_prompt("请用两句话介绍蓝鲸的食性和分类。")
print(result.metadata["response"])
for item in result.method_scores:
    print(item.method, item.risk, item.raw["cosine_similarities"])

## 结果解读与限制

向量相似不代表蕴含或事实支持；规范化后的 [0,1] 分数只用于同一配置下的比较。